# Cyclone Operations Lab

**Milan Tony · Data science take-home**

This notebook reads the supplied workbook, validates the five-minute timeline, explores the delivered analysis, and reviews the RAG prototype. Run it from the repository root after installing `requirements.txt`. The full reproducible model fitting lives in `Task1/task1_analysis.py`; run that script to regenerate all outputs.

> Interpretation boundary: shutdowns and root causes are inferred from sensors without operator ground truth.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from Task1.task1_analysis import prepare, periods
ROOT = Path.cwd()
assert (ROOT / "docs/source-data/data.xlsx").exists(), "Start Jupyter in the repository root"
plt.style.use("seaborn-v0_8-whitegrid")


## 1. Data preparation and quality

Parse numeric sensor readings while retaining counts of communication-error strings. Consolidate duplicate timestamps by median, then reindex every five minutes. Missing readings remain missing to avoid inventing operating states.


In [ ]:
raw = pd.read_excel(ROOT / "docs/source-data/data.xlsx")
data, quality = prepare(raw)
summary = pd.DataFrame({"metric": ["source rows", "start", "end", "missing timestamps", "duplicate timestamps"], "value": [quality["raw_rows"], quality["start"], quality["end"], quality["missing_timestamps"], quality["duplicate_timestamps"]]})
display(summary)
display(pd.Series(quality["missing_by_sensor"], name="missing samples").to_frame())


In [ ]:
display(data.describe(percentiles=[.01,.05,.5,.95,.99]).round(2))
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(data.corr(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(data.columns)), data.columns, rotation=70, ha="right")
ax.set_yticks(range(len(data.columns)), data.columns)
fig.colorbar(image, ax=ax); fig.tight_layout(); plt.show()


## 2. Shutdown evidence

A shutdown proxy requires both gas temperatures below 150 and absolute inlet draft below 30 for at least 30 consecutive minutes. The threshold is a transparent rule based on the separation between hot operation and near-ambient readings. Validate it with operating logs before treating it as a formal availability KPI.


In [ ]:
shutdowns = pd.read_csv(ROOT / "Task1/shutdown_periods.csv", parse_dates=["start", "end"])
print(f"Events: {len(shutdowns):,} | Detected hours: {shutdowns.duration_minutes.sum()/60:,.1f}")
display(shutdowns.nlargest(10, "duration_minutes"))
display(Markdown("![2018 sensor overview](Task1/plots/full_year.png)"))


## 3. Active operation states

Only complete, non-shutdown observations enter the clustering model. Six states were selected from candidates 3 through 6 by sampled silhouette score. State names describe sensor profiles; they do not assign confirmed faults or efficiency levels.


In [ ]:
selection = pd.read_csv(ROOT / "Task1/cluster_selection.csv")
clusters = pd.read_csv(ROOT / "Task1/clusters_summary.csv")
display(selection.round(3))
cols = ["state", "state_name", "samples", "frequency_fraction", "episodes", "median_duration_minutes", "Cyclone_Inlet_Gas_Temp_median", "Cyclone_Inlet_Draft_median"]
display(clusters[cols].round(2))
ax = clusters.plot.bar(x="state_name", y="frequency_fraction", legend=False, figsize=(10,4), color="#197da1")
ax.set_ylabel("Fraction of active samples"); ax.set_xlabel(""); plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()


## 4. State-relative anomalies

Within each state, robust median absolute deviations identify unusual sensor values. Consecutive flags become events. The selected plots and before/during measurements support hypotheses that still require maintenance records.


In [ ]:
anomalies = pd.read_csv(ROOT / "Task1/anomalous_periods.csv", parse_dates=["start", "end"])
print(f"Candidate events: {len(anomalies):,}")
display(anomalies.nlargest(10, "peak_score"))
display(Markdown((ROOT / "Task1/anomaly_cases.md").read_text(encoding="utf-8")[:6000]))


## 5. Twelve forecast horizons

The test split is chronological. Persistence predicts the value at the forecast origin; gradient boosting uses lag and rolling features available at that origin. The one-hour target is twelve five-minute steps ahead.


In [ ]:
metrics = pd.read_csv(ROOT / "Task1/forecast_metrics.csv")
display(metrics[metrics.horizon_steps.astype(str).eq("overall")].round(2))
for model, part in metrics[~metrics.horizon_steps.astype(str).eq("overall")].groupby("method"):
    plt.plot(part.horizon_steps.astype(int)*5, part.RMSE, marker="o", label=model)
plt.xlabel("Horizon (minutes)"); plt.ylabel("Held-out RMSE"); plt.legend(); plt.show()


## 6. Cited document retrieval

The supplied 11 PDFs produce 382 indexed chunks. Run `python Task2/prototype/rag_prototype.py --rebuild` before the next cell. This cell inspects retrieval and page provenance without loading the generator. For generated answers and excerpts, use the dashboard or prototype CLI.


In [ ]:
from Task2.prototype.rag_prototype import RAG
index_dir = ROOT / "Task2/prototype/index"
if (index_dir / "vectors.faiss").exists():
    rag = RAG(index_dir)
    evidence, similarity = rag.retrieve("How often does Linatex recommend internal cyclone inspections?")
    print("Top dense similarity:", round(similarity, 3))
    display(pd.DataFrame(evidence)[["document", "page", "text"]])
else:
    print("Build the local index using the command above, then rerun this cell.")


## 7. Decisions and limitations

- Confirm the 77 shutdown-proxy events against operator and maintenance logs.
- Monitor telemetry loss separately from mechanical downtime.
- Review the six state profiles with process engineers before naming operating conditions.
- Triage high-score anomaly cases using fan status, calibration records, and timestamps.
- The one-hour gradient boosting model improves RMSE from 22.68 to 19.48, but transition periods deserve a separate prospective evaluation.
- The RAG smoke test has two answerable page labels and does not establish production faithfulness.
